# objective :
- 1. Find out the list of most popular and liked genre
- 2. Create Model that finds the best suited Movie for one
user in every genre.
- 3. Find what Genre Movies have received the best and
worst ratings based on User Rating.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import necessary libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as pyplot
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/combined_data_1.txt", header = None, names=["CustId", "Rating"], usecols = [0,1])

In [ ]:
df.head()

,CustId,Rating
0,1:,NaN
1,1488844,3.0
2,822109,5.0
3,885013,4.0
4,30878,4.0


### **the null values right now do not need to be filled or dropped as we have null values only across movie ids so first we shall separate them**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24058263 entries, 0 to 24058262
Data columns (total 2 columns):
 #   Column  Dtype  
---  ------  -----  
 0   CustId  object 
 1   Rating  float64
dtypes: float64(1), object(1)
memory usage: 367.1+ MB


In [ ]:
df.shape

(24058263, 2)

In [ ]:
# total no. of movies

movie_count=df.isnull().sum()['Rating']
movie_count

np.int64(4499)

In [ ]:
# no. of customers
total_cust=df['CustId'].nunique()

In [ ]:
cust_count=total_cust-movie_count

In [ ]:
cust_count

np.int64(470758)

In [ ]:
df['Rating'].value_counts()

,count
Rating,
4.0,8085741
3.0,6904181
5.0,5506583
2.0,2439073
1.0,1118186


In [ ]:
# separate movie_id column
movie_id=None
movie_id_per_row=[]

for i in df['CustId']:
  if ':' in i:
    movie_id=int(i.replace(':',""))
  movie_id_per_row.append(movie_id)



In [ ]:
df['Movie_id']=movie_id_per_row

In [ ]:
df.head()

,CustId,Rating,Movie_id
0,1:,NaN,1
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1


In [ ]:
df.dropna(inplace=True)

In [ ]:
# check null values

df.isnull().sum()

,0
CustId,0
Rating,0
Movie_id,0


In [ ]:
# type casting

df['CustId']=df['CustId'].astype(int)

## Create a list of movies with very few ratings

In [ ]:
summary_cust=df.groupby('Movie_id')['Rating'].agg(['count'])

In [ ]:
summary_cust

,count
Movie_id,
1,547
2,145
3,2012
4,142
5,1140
...,...
4495,614
4496,9519
4497,714


In [ ]:
cust_benchmark = round(np.percentile(summary_cust['count'], 60, method = 'linear'))

In [ ]:
drop_cust_list = summary_cust[summary_cust['count']<cust_benchmark].index
drop_cust_list

Index([   1,    2,    4,    7,    9,   10,   11,   12,   13,   14,
       ...
       4480, 4481, 4486, 4487, 4491, 4494, 4495, 4497, 4498, 4499],
      dtype='int64', name='Movie_id', length=2699)

In [ ]:
# now we will keep a benchmark at 60th percentile

movie_benchmark = round(np.percentile(summary_cust['count'], 60, method = 'linear'))

In [ ]:
drop_movie_list = summary_cust[summary_cust['count']<movie_benchmark].index

In [ ]:
df.columns

Index(['CustId', 'Rating', 'Movie_id'], dtype='object')

In [ ]:
netflix_dataset = df[~df["Movie_id"].isin(drop_movie_list)]
netflix_dataset = df[~df["CustId"].isin(drop_cust_list)]

In [ ]:
title_df=pd.read_csv('/content/movies (1) (1).csv',encoding = "ISO-8859-1",usecols =[0,1,2])

In [ ]:
title_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
title_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  27278 non-null  int64 
 1   title    27278 non-null  object
 2   genres   27278 non-null  object
dtypes: int64(1), object(2)
memory usage: 639.5+ KB


In [ ]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.9 MB/s eta 0:00:00


In [ ]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

In [ ]:
# Reader --> it acts as a translator
# Dataset --> the converter
# SVD --> brain of model

In [ ]:
df.columns

Index(['CustId', 'Rating', 'Movie_id'], dtype='object')

In [ ]:
reader=Reader()

data=Dataset.load_from_df(netflix_dataset[['CustId', 'Rating', 'Movie_id']][:100000],reader)

In [ ]:
model=SVD()

In [ ]:
cross_validate(model,data,measures=['rmse'],cv=3)

{'test_rmse': array([18.32527761, 18.26001442, 18.2629269 ]),
 'fit_time': (1.7494142055511475, 1.1988365650177002, 1.2531588077545166),
 'test_time': (0.4340367317199707, 0.15264391899108887, 0.15065336227416992)}

-  test_rmse:  RMSE for each cross-validation fold. Lower is better.
-  Average RMSE: ≈ 18.28, meaning the model's average prediction error is about 18.28 units.
- fit_time: Time taken to train the model in each fold.
- test_time: Time taken to make predictions/evaluate the model.

##                   Recommendation

In [ ]:
title_df.columns

Index(['movieId', 'title', 'genres'], dtype='object')

## 1. Find out the list of most popular and liked genre

In [ ]:
genre_df = title_df['genres'].str.split('|').explode()

popular_genre = genre_df.value_counts()

print(popular_genre)

genres
Drama                 13344
Comedy                 8374
Thriller               4178
Romance                4127
Action                 3520
Crime                  2939
Horror                 2611
Documentary            2471
Adventure              2329
Sci-Fi                 1743
Mystery                1514
Fantasy                1412
War                    1194
Children               1139
Musical                1036
Animation              1027
Western                 676
Film-Noir               330
(no genres listed)      246
IMAX                    196
Name: count, dtype: int64


In [ ]:
df.columns

Index(['CustId', 'Rating', 'Movie_id'], dtype='object')

In [ ]:
title_df.rename(columns={'movieId':'Movie_id'},inplace=True)

In [ ]:
title_df.columns

Index(['Movie_id', 'title', 'genres'], dtype='object')

In [ ]:
# Most popular genres = genres with the highest number of ratings

genre_popular = (
    title_df
    .assign(genres=title_df['genres'].str.split('|'))
    .explode('genres')
    .merge(
        netflix_dataset[['Movie_id', 'Rating']],
        on='Movie_id',
        how='inner'
    )
    .groupby('genres')
    .agg(
        total_ratings=('Rating', 'count'),
        average_rating=('Rating', 'mean')
    )
    .sort_values('total_ratings', ascending=False)
)

print(genre_popular)

             total_ratings  average_rating
genres                                    
Drama             11439918        3.581610
Comedy             8444143        3.601335
Romance            3858665        3.605981
Thriller           3325255        3.544709
Crime              2930915        3.539043
Action             2787471        3.605826
Horror             2644360        3.733706
Adventure          2264939        3.681398
Children           1522941        3.661372
Sci-Fi             1476093        3.614397
Fantasy            1096655        3.643818
Musical            1064134        3.734470
Documentary         930385        3.513773
Mystery             872762        3.530386
Animation           629950        3.709956
Film-Noir           484985        3.581766
Western             473504        3.645374
War                 397718        3.528334
IMAX                 24964        3.825989


- Drama is the most popular genre based on user ratings, while IMAX is the most liked genre with the highest average rating.

## Create Model that finds the best suited Movie for one user in every genre

In [ ]:
!free -h

# Create Model that finds the best suited Movie for one user in every genre

In [ ]:
netflix_dataset.head()

,CustId,Rating,Movie_id
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1


In [ ]:
user_id = 1488844

user_ratings = netflix_dataset[
    netflix_dataset['CustId'] == user_id
]

In [ ]:
user_ratings = user_ratings.sort_values(
    'Rating',
    ascending=False
)

user_ratings.head()

,CustId,Rating,Movie_id
22278921,1488844,5.0,4227
22662492,1488844,5.0,4302
22719013,1488844,5.0,4306
20333360,1488844,5.0,3864
20774899,1488844,5.0,3925


In [ ]:
user_movies = user_ratings.merge(
    title_df[['Movie_id', 'title', 'genres']],
    on='Movie_id',
    how='left'
)

In [ ]:
user_movies = (
    user_movies
    .assign(genres=user_movies['genres'].str.split('|'))
    .explode('genres')
)

In [ ]:
best_movies = (
    user_movies
    .sort_values('Rating', ascending=False)
    .groupby('genres')
    .first()
    .reset_index()
)

best_movies = best_movies[
    ['genres', 'title', 'Rating']
]

print(best_movies)

         genres                                      title  Rating
0        Action                            Daylight (1996)     5.0
1     Adventure                     Doctor Dolittle (1967)     5.0
2     Animation                               Shrek (2001)     5.0
3      Children                     Doctor Dolittle (1967)     5.0
4        Comedy                        Hear My Song (1991)     5.0
5         Crime                      Coogan's Bluff (1968)     5.0
6   Documentary        Kestrel's Eye (Falkens Ã¶ga) (1998)     4.0
7         Drama                      At Close Range (1986)     5.0
8       Fantasy  Darby O'Gill and the Little People (1959)     5.0
9     Film-Noir              Sweet Smell of Success (1957)     5.0
10       Horror                       Event Horizon (1997)     5.0
11      Musical                     Doctor Dolittle (1967)     5.0
12      Mystery                             Amistad (1997)     5.0
13      Romance                    Big Country, The (1958)    

 ## Personalized recommendation system that identifies the best-suited movie for a user in each genre based on their past ratings.

Find what Genre Movies have received the best and
worst ratings based on User Rating.

In [ ]:
# Average rating for each movie
movie_rating = netflix_dataset.groupby('Movie_id')['Rating'].mean()

# Map ratings to movie details
title_df['avg_rating'] = title_df['Movie_id'].map(movie_rating)

# Remove movies with no rating
temp = title_df.dropna(subset=['avg_rating']).copy()

# Remove "(no genres listed)"
temp = temp[temp['genres'] != '(no genres listed)']

# Split genres
genre_rating = (
    temp
    .assign(genres=temp['genres'].str.split('|'))
    .explode('genres')
    .groupby('genres')['avg_rating']
    .mean()
    .sort_values(ascending=False)
)

print("Best Rated Genre:")
print(genre_rating.head(1))

print("\nWorst Rated Genre:")
print(genre_rating.tail(1))

Best Rated Genre:
genres
IMAX    3.529245
Name: avg_rating, dtype: float64

Worst Rated Genre:
genres
War    3.161334
Name: avg_rating, dtype: float64


## **IMAX movies received the highest average user rating (3.53), while War movies received the lowest average rating (3.16).